In [114]:
import json
from typing import List

In [136]:
data = json.load(open('./vocabulary/en_graph.json', 'r'))

In [116]:
data.keys()

dict_keys(['vowel', 'consonant', 'composed_vowel', 'composed_consonant', 'mixed', 'many', 'past', 'prefix', 'suffix', 'dictionary', 'exept', 'replace', 'hotword'])

In [117]:
def get_last_item(word: str, pattern: str):
    length_item = len(pattern)
    start_check = len(word) - length_item
    if word[start_check: ] == pattern:
        return word[:start_check ], pattern
    return None

In [118]:
def get_first_item(word: str, pattern: str):
    length_item = len(pattern)
    if word[0: length_item] == pattern:
        return pattern, word[length_item: ]
    return None

In [119]:
def get_by_prev(word: str, patterns: List[dict]):
    for key in patterns.keys():
        items = get_last_item(word, key)
        if items is not None:
            for pattern in patterns[key]:
                specific_items = get_last_item(items[0], pattern)
                if specific_items is not None:
                    return items[0], [*items[1]]
            return items[0], [items[1]]
    return None


In [120]:
get_by_prev("played", data['past'])

('play', ['ed'])

In [139]:
def get_prefix(word: str, patterns: List[dict]):
    for key in patterns:
        items = get_first_item(word, key)
        if items is not None:
            return [*items[0]], items[1]
    return [], word

In [126]:
def get_suffix(word: str, patterns: List[dict]):
    for key in patterns.keys():
        items = get_last_item(word, key)
        if items is not None:
            return items[0], [*items[1]]
    return None

In [123]:
def get_last(word: str, suffix_patterns: List[dict], past_patterns: List[dict], quantity_patterns: List[dict]):
    suffixes = []
    count = 0

    last_items = None
    last_items = get_by_prev(word, quantity_patterns)
    if last_items is not None:
        word = last_items[0]
        suffixes.append(last_items[1])
    
    while True:
        last_items = None
        last_items = get_suffix(word, suffix_patterns)
        if last_items is not None:
            word = last_items[0]
            suffixes.append(last_items[1])
            count = 0
        else:
            count += 1

        last_items = get_by_prev(word, past_patterns)
        if last_items is not None:
            word = last_items[0]
            suffixes.append(last_items[1])
            count = 0
        else:
            count += 1

        if count > 2:
            break

    return word, suffixes

In [124]:
word, suffixes = get_last("interestedly", data['suffix'], data['past'], data['many'])

In [141]:
get_prefix(word, data['prefix'])

([], 'interest')

In [137]:
get_suffix("interestingly", data['suffix'])

('interesting', ['l', 'y'])

In [ ]:
get_prefix

In [24]:
def get_last_list(word: str, patterns: List[str]):
    for pattern in patterns:
        items = get_last_item(word, pattern)
        if items is not None:
            return items
    return None

In [25]:
def get_last_split_item(word: str, patterns: dict):
    for pattern in patterns.keys():
        items = get_last_item(word, pattern)
        if items is not None:
            pattern_items = get_last_list(items[0], list(patterns[pattern]))
            if pattern_items is not None:
                return items[0], [*items[1]]
    return None

In [26]:
def get_last_auto_split(word, patterns: dict):
    print(patterns)
    for pattern in patterns.keys():
        items = get_last_item(word, pattern)
        if items is not None:
            return items[0], patterns[pattern]
    return None

In [27]:
def get_first_item(word: str, patterns: List[str]):
    for item in patterns:
        if item in word:
            first_item = word[:len(item)]
            if first_item == item:
                return first_item, word[len(item):]
    return None

In [28]:
def word2graphemes(text: str, patterns: List, n_grams: int = 4):
        if len(text) == 1:
            if text in patterns:
                return [text]
            return ["<unk>"]
        graphemes = []
        start = 0
        if len(text) - 1 < n_grams:
            n_grams = len(text)
        num_steps = n_grams
        while start < len(text):
            found = True
            item = text[start:start + num_steps]
            
            if item in patterns:
                graphemes.append(item)
            elif num_steps == 1:
                graphemes.append("<unk>")
            else:
                found = False

            if found:
                start += num_steps
                if len(text[start:]) < n_grams:
                    num_steps = len(text[start:])
                else:
                    num_steps = n_grams
            else:
                num_steps -= 1

        return graphemes

In [29]:
def exception_handle(word: str, patterns: List[dict]):
    for item in patterns.keys():
        if item == word:
            return patterns[item]
    return []

In [17]:
exception_handle("soldier", data['exception']['word'])

dict_keys(['often', 'prayer', 'listen', 'soldier'])
often
prayer
listen
soldier


['s', 'o', 'di', 'e', 'r']

In [30]:
first_patterns = data['vowel'] + data['consonant'] + data['composed_vowel'] + data['composed_consonant']
first_patterns = sorted(first_patterns, key=len, reverse=True)

stride_patterns = first_patterns + data['mixed']

In [34]:
word = input()

first_text = ''
last_items = []

last_item_outputs = get_last_split_item(word, data['last_split_item'])

if last_item_outputs is not None:
    word = last_item_outputs[0]
    last_items += last_item_outputs[1]

tmp = []
while True:
    last_item_outputs = get_last_list(word, data['last_special'])
    if last_item_outputs is not None:
        word = last_item_outputs[0]
        tmp.append(last_item_outputs[1])
    else:
        break
tmp = [tmp[i] for i in range(len(tmp) - 1, -1, -1)]
last_items += tmp
first_items = get_first_item(word, first_patterns)

if first_items is not None:
    first_text = first_items[0]
    word = first_items[1]

item_graphemes = word2graphemes(word, stride_patterns)

graphemes = [first_text] + item_graphemes + last_items
print(graphemes)

['l', 'i', 's', 't']
